In [1]:
####################################
# imports
####################################
from pathlib import Path
import json
import pandas as pd

In [2]:
####################################
# local file paths
####################################
input_candidates = [
    Path("working_files/data/maknuune-v1.0.1_cleaned_predictions.csv"),
    Path("data/maknuune-v1.0.1_cleaned_predictions.csv"),
    Path("maknuune-v1.0.1_cleaned_predictions.csv"),
]

input_file = next((path for path in input_candidates if path.exists()), None)
if input_file is None:
    searched = "\n".join(str(path) for path in input_candidates)
    raise FileNotFoundError(f"Could not find the cleaned predictions CSV file. Searched:\n{searched}")

output_file = input_file.with_name("maknuune-v1.0.1_cleaned_predictions_top_5.csv")
print("Input file:", input_file)
print("Output file:", output_file)

Input file: data/maknuune-v1.0.1_cleaned_predictions.csv
Output file: data/maknuune-v1.0.1_cleaned_predictions_top_5.csv


In [3]:
####################################
# load csv into df
####################################
df = pd.read_csv(input_file)
print("Original DataFrame loaded.")
print("Shape:", df.shape)
print("Columns:", list(df.columns))

Original DataFrame loaded.
Shape: (36302, 8)
Columns: ['arabizi', 'arabic_harakat', 'arabic_stripped', 'harakat_predictions_softmax_90', 'harakat_top_prediction', 'harakat_top_probability', 'harakat_top_log_probability', 'harakat_prediction_done_at']


In [4]:
####################################
# harakat prediction extraction function
####################################
def get_top_harakat_predictions(predictions_text, top_n=5):
    if pd.isna(predictions_text):
        return [pd.NA] * top_n

    predictions = json.loads(predictions_text)
    predictions = sorted(
        predictions,
        key=lambda prediction: prediction.get("probability", 0),
        reverse=True,
    )

    top_predictions = [prediction.get("text", pd.NA) for prediction in predictions[:top_n]]
    while len(top_predictions) < top_n:
        top_predictions.append(pd.NA)

    return top_predictions

In [5]:
####################################
# top 5 prediction execution and testing
####################################
top_5_predictions = df["harakat_predictions_softmax_90"].apply(get_top_harakat_predictions)

for prediction_number in range(1, 6):
    df[f"harakat_prediction_{prediction_number}"] = top_5_predictions.apply(
        lambda predictions: predictions[prediction_number - 1]
    )

print("\nPreview of cleaned prediction data:")
print(df.head(20))

for col in [f"harakat_prediction_{prediction_number}" for prediction_number in range(1, 6)]:
    print(f"\nMissing values in {col}:")
    print(df[col].isna().sum())


Preview of cleaned prediction data:
                                 arabizi  \
0                                  2abad   
1                                  2ibre   
2                                  2ibar   
3                   qadd khurum 2il2ibre   
4                       2ibritil 3ajuuze   
5             2il2ibre ghalbat 2il7aayik   
6                            min 2ibirto   
7                                 2abaat   
8                                   baat   
9                      7atto ta7it baato   
10                                   2ab   
11                                 2ibwe   
12                               2aabaa2   
13                             2abbahaat   
14                            2abu 7maar   
15  mish ma3ruuf qar3it 2abuuha min ween   
16                            2abu surra   
17                         2abu kammuune   
18                         2abul 3urreef   
19                         2abul 3akkaat   

                           arabic_hara

In [6]:
####################################
# save results
####################################
df.to_csv(output_file, index=False)
print("\nDone.")
print("Cleaned predictions top 5 CSV saved to:", output_file)


Done.
Cleaned predictions top 5 CSV saved to: data/maknuune-v1.0.1_cleaned_predictions_top_5.csv
